## Libraries

In [11]:
# Import needed libraries

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix
from sklearn.metrics import roc_auc_score, roc_curve

import os
import joblib


## Figure Theme

In [ ]:
# Classical navy/cream theme so every figure matches the presentation deck.
import matplotlib as mpl

NAVY   = '#223343'   # slide background (theme accent1)
CREAM  = '#DFD1C2'   # text / axes      (theme lt1)
CREAMD = '#B5A093'   # secondary / spines (theme accent6)
GREEN  = '#66c2a5'   # safe / primary bars
ORANGE = '#fc8d62'   # danger / highlight

mpl.rcParams.update({
    'figure.facecolor':  NAVY,
    'axes.facecolor':    NAVY,
    'savefig.facecolor': NAVY,
    'font.family':       'serif',
    'font.serif':        ['Georgia', 'Cambria', 'Times New Roman', 'DejaVu Serif'],
    'text.color':        CREAM,
    'axes.labelcolor':   CREAM,
    'axes.titlecolor':   CREAM,
    'xtick.color':       CREAM,
    'ytick.color':       CREAM,
    'axes.edgecolor':    CREAMD,
})

## Data Load

In [14]:
data_path = r"C:\Users\k_spa\OneDrive\Documents\secondary+mushroom+dataset\MushroomDataset\MushroomDataset\secondary_data.csv"
mushrooms = pd.read_csv(data_path, sep=';')

print(f"Records: {len(mushrooms):,}")
print(f"Columns: {len(mushrooms.columns)}")
print(mushrooms.head())

Records: 61,069
Columns: 21
  class  cap-diameter cap-shape cap-surface cap-color does-bruise-or-bleed  \
0     p         15.26         x           g         o                    f   
1     p         16.60         x           g         o                    f   
2     p         14.07         x           g         o                    f   
3     p         14.17         f           h         e                    f   
4     p         14.64         x           h         o                    f   

  gill-attachment gill-spacing gill-color  stem-height  ...  stem-root  \
0               e          NaN          w        16.95  ...          s   
1               e          NaN          w        17.99  ...          s   
2               e          NaN          w        17.80  ...          s   
3               e          NaN          w        15.77  ...          s   
4               e          NaN          w        16.53  ...          s   

  stem-surface stem-color veil-type veil-color has-ring ri

## Data Exploration

In [17]:
# Checks missing values in each column. The highest value of missing data is veil-type and spore color.
print("\nMISSING DATA\n")
print(mushrooms.isnull().sum().sort_values(ascending=False))

# Prints the percentage of missing data per column.
missing_pct = (mushrooms.isnull().mean() * 100).sort_values(ascending=False)     
print("\nMISSING DATA (% of rows)\n")                                            
print(missing_pct[missing_pct > 0].round(1))                                     

# Prints duplicated number of rows which is 146.                             
print("\nExact duplicate rows:", mushrooms.duplicated().sum())                   

# Checks the proportion of edible vs poisonous mushrooms. This shows over half the mushrooms are poisonous in the dataset
print("\nEDIBILITY vs. POISON\n")
print(mushrooms['class'].value_counts())
print(mushrooms['class'].value_counts(normalize=True).mul(100).round(1))         


MISSING DATA

veil-type               57892
spore-print-color       54715
veil-color              53656
stem-root               51538
stem-surface            38124
gill-spacing            25063
cap-surface             14120
gill-attachment          9884
ring-type                2471
class                       0
stem-color                  0
habitat                     0
has-ring                    0
stem-width                  0
cap-diameter                0
stem-height                 0
gill-color                  0
does-bruise-or-bleed        0
cap-color                   0
cap-shape                   0
season                      0
dtype: int64

MISSING DATA (% of rows)

veil-type            94.8
spore-print-color    89.6
veil-color           87.9
stem-root            84.4
stem-surface         62.4
gill-spacing         41.0
cap-surface          23.1
gill-attachment      16.2
ring-type             4.0
dtype: float64

Exact duplicate rows: 146

EDIBILITY vs. POISON

class
p    33888

## Data Clean

In [20]:
# Grabs the top 4 missing data values which are in the 50K range and creates new colum called empty_values
empty_values = ['stem-root', 'veil-type', 'veil-color', 'spore-print-color']
mushrooms = mushrooms.drop(columns=empty_values)

# For the columns with a few missing values, they are being filled with "unknown" instead of tossing the row.
mushrooms = mushrooms.fillna('unknown')
print("\nMissing values after cleaning:", mushrooms.isnull().sum().sum())


Missing values after cleaning: 0


## Data Exploration Visualizations

In [ ]:
# Edible vs. Poisonous Mushrooms bar chart shows the value of each in this dataset.
# This dataset shows that there are 33,888 mushrooms that are labeled poisonous and
# 27,181 that are labeled edible.
edibility = mushrooms['class'].map({'p': 'Poisonous', 'e': 'Edible'})
counts = edibility.value_counts()

# Creates the bar chart. Green = poisonous, orange = edible (matches the deck).
fig, ax = plt.subplots(figsize=(7.5, 5.2))
bars = ax.bar(counts.index, counts.values, color=[GREEN, ORANGE], width=0.6)
ax.set_title('Number of Edible vs Poisonous Mushrooms')
ax.set_xlabel('Mushroom Type')
ax.set_ylabel('Number of Mushrooms')
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

# Loop to add how many mushrooms are either poisonous or edible on top of the bar chart.
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, height + 300,
            f'{height:,}', ha='center', color=CREAM)
plt.show()


In [ ]:
# Boxplot showing the cap diameter for poisonous and edible mushrooms. In this instance,
# edible mushrooms tend to have a larger median cap size than poisonous mushrooms.
edibility = mushrooms['class'].map({'p': 'Poisonous', 'e': 'Edible'})

fig, ax = plt.subplots(figsize=(7.5, 5.2))
sns.boxplot(
    data=mushrooms.assign(edibility=edibility),
    x='edibility',
    y='cap-diameter',
    showfliers=False,
    palette={'Poisonous': GREEN, 'Edible': ORANGE},
    ax=ax
)

# Style the box/whisker lines so they read on the dark theme.
for line in ax.lines:
    line.set_color(CREAM)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

ax.set_title('Cap Diameter by Class')
ax.set_xlabel('Mushroom Type')
ax.set_ylabel('Cap Diameter (cm)')
plt.show()


## Model Data Prep

In [28]:
# Predicting y. 1 is poisonous, and 0 is edible.
y = (mushrooms['class'] == 'p').astype(int)

# X is going to be everything except the answer column which is class. I am calling get_dummies to
# take categories with text and turn them into binary for my model.
X = pd.get_dummies(mushrooms.drop(columns=['class']))

# Splitting into a 75/25 train/test split. Confirmed over 45,000 training rows,
# and over 15,000 testing rows.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print(f"\nTrain # of rows: {len(X_train):,}   Test # of rows: {len(X_test):,}")


Train # of rows: 45,801   Test # of rows: 15,268


## Training & Comparing Models

In [31]:
# Includes three models to train and then compare:
# Logistic regression, decision tree, and random forest.

# The first model we are training is logistic regression. I am getting 82.6% accuracy and 
# it is classyfying 83% poisonous mushrooms correctly.
l_reg = LogisticRegression(max_iter=1000)
l_reg.fit(X_train, y_train)
l_reg_predictions = l_reg.predict(X_test)

print("\nLOGISTIC REGRESSION\n")
print(f"Accuracy: {accuracy_score(y_test, l_reg_predictions):.3f}")
print(f"Recall: {recall_score(y_test, l_reg_predictions):.3f}")

# Decision Tree model training. I am only using a depth size of 8 which will impact my recall.
# I am getting 82% accuracy and it is classyfing 74% poisonous mushrooms correctly.
d_tree = DecisionTreeClassifier(max_depth=8, random_state=42)
d_tree.fit(X_train, y_train)
d_tree_predictions = d_tree.predict(X_test)

print("\nDECISION TREE\n")
print(f"Accuracy: {accuracy_score(y_test, d_tree_predictions):.3f}")
print(f"Recall: {recall_score(y_test, d_tree_predictions):.3f}")

# Random Forest model training with 100 estimators. I am getting perfect accuracy and recall.
r_forest = RandomForestClassifier(n_estimators=100, random_state=42)
r_forest.fit(X_train, y_train)
r_forest_predictions = r_forest.predict(X_test)

print("\nRANDOM FOREST\n")
print(f"Accuracy: {accuracy_score(y_test, r_forest_predictions):.3f}")
print(f"Recall: {recall_score(y_test, r_forest_predictions):.3f}")


LOGISTIC REGRESSION

Accuracy: 0.826
Recall: 0.834

DECISION TREE

Accuracy: 0.824
Recall: 0.739

RANDOM FOREST

Accuracy: 1.000
Recall: 1.000


## Modeling Visualizations

In [ ]:
# Confusion matrix to show which of the guesses were right and wrong. There were
# 8,487 that were classified as poisonous and 6,781 that were edible.
from matplotlib.colors import LinearSegmentedColormap

c_matrix = confusion_matrix(y_test, r_forest_predictions)

# Navy -> green colormap so the heatmap matches the deck theme.
theme_cmap = LinearSegmentedColormap.from_list('navy_green', [NAVY, '#2f5d4a', GREEN])

# Creates the confusion matrix.
fig, ax = plt.subplots(figsize=(6.4, 5.4))
sns.heatmap(c_matrix, annot=True, fmt=',d', cmap=theme_cmap,
            xticklabels=['Edible', 'Poisonous'], yticklabels=['Edible', 'Poisonous'],
            annot_kws={'color': CREAM, 'fontsize': 14, 'fontweight': 'bold'},
            linewidths=1, linecolor=NAVY, cbar=True, ax=ax)
ax.set_title('Random Forest - Confusion Matrix')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')

# Style the colorbar to the theme.
cbar = ax.collections[0].colorbar
cbar.ax.yaxis.set_tick_params(color=CREAM)
cbar.outline.set_edgecolor(CREAMD)
plt.setp(plt.getp(cbar.ax.axes, 'yticklabels'), color=CREAM)

# Confirms that there were no poisonous mushrooms that were called edible.
print("\nPoisonous Mushrooms Called Edible:", c_matrix[1, 0])
plt.show()


In [ ]:
# Probability the Random Forest gives to the poisonous class for each mushroom.
poison_probability = r_forest.predict_proba(X_test)[:, 1]

# AUC summarizes how well the model ranks poisonous above edible. The model hitting 1
# shows a perfect score fit.
auc = roc_auc_score(y_test, poison_probability)
print(f"\nRandom Forest - ROC-AUC: {auc:.3f}")

# Checking multiple cutoffs 0.50 to 0.05.
# For each of the cutoffs I am checking to see how many poisonous mushrooms are caughts and how many
# edible ones are flagged as poisonous.
print("\nThreshold Tuning\n")

edible_count = (y_test == 0).sum()
cutoffs = [0.50, 0.30, 0.20, 0.10, 0.05]

# Loops through each of the 5 defined cutoffs to check how many poisonous mushrooms are caught
# and how many are flagged as poisonous but are actually edible. The strictest cutoff is 0.05 and that
# caught 118 mushrooms that were marked as poisonous but were actually edible.
for cutoff in cutoffs:
    preds = (poison_probability >= cutoff).astype(int)
    caught = recall_score(y_test, preds)
    misses = ((preds == 0) & (y_test == 1)).sum()      # poisonous called edible
    false_alarms = ((preds == 1) & (y_test == 0)).sum()  # edible called poisonous
    print(f"Cutoff - {cutoff:.2f}: Caught Poisonous - {caught:.3f}, "
          f"Misses - {misses}, Edibles Flagged as Poisonous - {false_alarms}")

# ROC curve shows the tradeoff between catching poisonous mushrooms
# and false alarms across every possible cutoff.
fpr, tpr, thresholds = roc_curve(y_test, poison_probability)

fig, ax = plt.subplots(figsize=(6.4, 5.4))
ax.plot(fpr, tpr, color=GREEN, linewidth=2, label=f'Random Forest (AUC = {auc:.3f})')
ax.plot([0, 1], [0, 1], '--', color=CREAMD, label='Random guessing')
ax.set_xlabel('Edible Flagged as Poisonous')
ax.set_ylabel('Poisonous Caught')
ax.set_title('ROC Curve - Safety Tradeoff')
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
legend = ax.legend(loc='lower right', facecolor=NAVY, edgecolor=CREAMD)
for text in legend.get_texts():
    text.set_color(CREAM)
plt.show()


## New Model For Novice Forager

In [39]:
# Since some of the traits may be hard for a novice forager to identify,
# below I created a new variable to store only easy to identify traits.
beginner_traits = ['cap-diameter', 'stem-height', 'stem-width', 'cap-shape','cap-color', 'does-bruise-or-bleed', 'stem-color',
                    'has-ring', 'habitat', 'season']

# Created a new 75/25 train/test split with only the 10 features listed above to see how well it still predicts
# if we get rid of the other traits.
X_easy = pd.get_dummies(mushrooms[beginner_traits])
Xe_train, Xe_test, ye_train, ye_test = train_test_split(
    X_easy, y, test_size=0.25, random_state=42
)

beginner_model = RandomForestClassifier(n_estimators=200, random_state=42)
beginner_model.fit(Xe_train, ye_train)
beginner_predictions = beginner_model.predict(Xe_test)

# Results show accuracy is still high at 99% and recall at 99%.
print("\nModel for Beginner Traits\n")
print(f"Accuracy: {accuracy_score(ye_test, beginner_predictions):.3f}")
print(f"Poisonous caught (recall): {recall_score(ye_test, beginner_predictions):.3f}")


Model for Beginner Traits

Accuracy: 0.989
Poisonous caught (recall): 0.992


## Important Traits

In [ ]:
# Using feature importance for my Random Forest model to see which features it relied on the most.
importances = pd.Series(r_forest.feature_importances_, index=X.columns)
top_10 = importances.sort_values(ascending=False).head(10)

# Highlight the three strongest size traits in orange, the rest in green.
top_asc = top_10.sort_values()
size_traits = ['stem-width', 'stem-height', 'cap-diameter']
bar_colors = [ORANGE if name in size_traits else GREEN for name in top_asc.index]

fig, ax = plt.subplots(figsize=(9, 6))
top_asc.plot(kind='barh', color=bar_colors, ax=ax)
ax.set_title('Top 10 Most Important Features')
ax.set_xlabel('Importance')
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

# The results show that stem-width, stem-height, and cap-diameter were the most important three.
print("\nTop Features\n")
print(top_10)
plt.show()
